# Redrob Candidate Ranking — Phase 1: Data Analysis & Label Discovery

**Goal:** understand the 100K candidate pool and *reverse-engineer the relevance-tier rubric* the hidden ground truth uses.

The scoring spec leaks the structure:
- P@10 = "fraction of top-10 that are **relevant (tier 3+)**" → labels are a **tier scale**.
- Docs reference **Tier-5** candidates and **~80 honeypots** (subtly impossible profiles; >10% in top-100 = disqualified).

So our labels are **tier 0–5** (5 = perfect fit) plus a **honeypot/exclude** flag. There is no label field in the data — we derive it from the JD's fit-theory.

In [ ]:
import json, collections, datetime as dt, re
import numpy as np, pandas as pd

DATA = 'data/candidates.jsonl'
TODAY = dt.date(2026, 6, 29)
TOKEN_RE = re.compile('[a-z0-9.+#-]+')   # word tokenizer (no substring false-positives)

# domain constants derived from the JD
INDIA_CITIES = {'bangalore','bengaluru','pune','noida','hyderabad','mumbai','delhi','gurgaon','gurugram','chennai','kolkata','ncr','new delhi'}
JD_CITIES = {'pune','noida','hyderabad','mumbai','delhi','gurgaon','gurugram','ncr','new delhi','bangalore','bengaluru'}
CONSULTING = {'tcs','tata consultancy','infosys','wipro','accenture','cognizant','capgemini','hcl','tech mahindra','ltimindtree','mindtree','mphasis','dxc'}
INDIA_EMPLOYER_HINTS = CONSULTING | {'flipkart','swiggy','zomato','ola','paytm','razorpay','phonepe','freshworks','zoho','meesho','myntra','zerodha','groww','delhivery','makemytrip','inmobi','jio','reliance','hdfc','icici','axis bank','redrob','sarvam','krutrim'}
SERVICE_INDS = {'it services','consulting','management consulting','outsourcing','staffing','bpo','business process outsourcing','professional services'}
PRODUCT_INDS = {'software','fintech','e-commerce','saas','ai/ml','food delivery','edtech','adtech','transportation','insurance tech'}
RETRIEVAL_EVIDENCE = ['recommendation system','ranking system','ranking model','search relevance','retrieval system','semantic search','vector search','recsys','personalization','learning to rank','information retrieval','embedding','candidate ranking','relevance model','matching system','search ranking','hybrid search','bm25','reranking','retrieval quality']
VECTOR_INFRA = ['pinecone','weaviate','qdrant','milvus','opensearch','elasticsearch','faiss','vector database','vector db','hybrid search','bm25','ann index','hnsw','index refresh','embedding drift','retrieval-quality regression','retrieval quality regression']
PROD_EVIDENCE = ['production','deployed','in prod','real users','at scale','serving','latency','a/b test','ab test','online experiment','index refresh','embedding drift','regression in production','production system']
EVAL_EVIDENCE = ['ndcg','ndcg@','mrr','mrr@','mean reciprocal','map@','mean average precision','precision@','recall@','offline metric','offline-to-online','offline to online','a/b test','ab test','evaluation framework','eval framework','ranking evaluation','relevance evaluation']
PYTHON_EVIDENCE = ['python','pandas','numpy','scipy','sklearn','scikit-learn','pytest','fastapi','pydantic','asyncio','sqlalchemy','pyarrow']
NICE_TO_HAVE = ['lora','qlora','peft','fine-tun','fine tuning','learning to rank','xgboost','lambdamart','hr-tech','recruiting tech','marketplace','distributed systems','large-scale inference','inference optimization','open source','opensource','github contribution','contributed to']
FRAMEWORK_DEMO = ['langchain','llamaindex','prompt engineering','tutorial','demo app','toy project','how i used','blog post']
EXTERNAL_VALIDATION = ['open source','opensource','github','publication','paper','conference','talk','speaker','patent','kaggle','maintainer']
ML_EVIDENCE = ['machine learning','deep learning','neural network','model training','trained a model','classifier','regression model','recommendation','ranking','embedding','natural language',' nlp','computer vision','predictive model','ml model','ml pipeline','feature engineering','xgboost','pytorch','tensorflow','fine-tun','llm','transformer','clustering']
LLM_RECENT = ['langchain','llamaindex','openai api','gpt-4','gpt4','prompt engineering','chatgpt']
RESEARCH_HINT = ['research scientist','phd','postdoc','research fellow','research lab','academic','publication']
NONENG_TITLES = ['marketing','sales','recruiter','human resources','hr manager','product manager','designer','accountant','operations','content writer','customer success','business analyst','project manager']
ML_TITLE = ['ml engineer','machine learning','ai engineer','applied scientist','data scientist','research engineer','nlp engineer','search engineer','ranking engineer','mlops']
CV_SPEECH = ['image classification','object detection','speech recognition','tts','computer vision','robotics','ocr','asr','segmentation','pose estimation']
AI_SKILL_KW = {'rag','retrieval','embedding','embeddings','vector','faiss','pinecone','weaviate','qdrant','milvus','llm','llms','nlp','fine-tuning llms','lora','qlora','peft','ranking','recommendation','transformer','semantic search','information retrieval','bm25'}
# verified assessment skills that indicate genuine retrieval/IR/LLM competency (the rare, hard-to-fake ones)
IR_ASSESS_TOKENS = ['bm25','embedding','faiss','haystack','fine-tun','elasticsearch','retrieval','semantic','vector','rag','llm','hugging face','sentence transform','information retrieval']

# Explicit JD scoring weights. Must-have production retrieval/search/eval signals dominate;
# nice-to-haves stay modest; anti-fit patterns can materially lower a profile.
STRUCT_WEIGHTS = {
    'yoe_target': 1.0,
    'yoe_greater_target': 0.7,
    'yoe_slightly_below_target': 0.6,
    'yoe_below_target': 0.3,
    'yoe_far_below_target': 0.0,
    'applied_ml': 1.0,
    'python_strong': 1.5,
    'python_some': 0.7,
    'ir_assess_strong': 1.5,
    'ir_assess_some': 0.7,
    'retrieval_prod_strong': 1.5,
    'retrieval_prod_some': 0.8,
    'vector_infra_strong': 1.5,
    'vector_infra_some': 0.8,
    'ranking_eval': 1.5,
    'product_context': 0.4,
    'product_heavy_career': 0.5,
    'service_heavy_with_product': 0.25,
    'profile_complete_high': 1.00,
    'profile_complete_mid': 0.80,
    'profile_complete_low': 0.50,
    'profile_complete_floor': 0.10,
    'assessment_quality_high': 1.00,
    'assessment_quality_mid': 0.70,
    'assessment_quality_low': 0.40,
    'ml_title': 0.3,
    'nice_each': 0.45,
    'nice_cap': 1.6,
    
}
STRUCT_PENALTIES = {
    'title_chaser': -0.8,
    'framework_demo_only': -0.8,
    'service_only_career': -0.1,
    'consulting_only': -2.0,
    'cv_speech_without_nlp': -1.5,
    'closed_proprietary_no_validation': -0.5,
    'research_without_prod': -0.8,
    'recent_llm_only': -0.8,
}
LOCATION_STRUCT_WEIGHTS = {
    'jd_city': 0.6,
    'india_relocate': 0.4,
    'india': 0.1,
    'outside_relocate': 0.0,
    'outside_no_relocate': -0.7,
}
# tool -> earliest plausible year (anachronism / honeypot detection)
TOOL_BIRTH = {'lora':2021,'qlora':2023,'rag':2020,'langchain':2022,'chatgpt':2022,'gpt-4':2023,'llama':2023,'pinecone':2021,'qdrant':2021,'bge':2023,'vllm':2023,'mistral':2023,'stable diffusion':2022,'peft':2022}

def lc(s): return (s or '').lower()
def parse_date(s):
    try: return dt.date.fromisoformat(s)
    except Exception: return None
def tokens(text): return set(TOKEN_RE.findall(text))
def tool_present(tool, toks): return all(pt in toks for pt in tool.split())
print('setup ready')

setup ready


## Step 0 — Confirm there is no hidden ground-truth label

If a `tier`/`label`/`honeypot` field existed we'd just use it. Enumerate **every key-path** across all 100K records and flag anything label-like.

In [107]:
SUSPICIOUS = ('tier','label','relevance','honeypot','trap','ground','truth','quality','target','gold')
keypaths = collections.Counter()
susp = collections.defaultdict(collections.Counter)
def walk(o, p=''):
    if isinstance(o, dict):
        for k,v in o.items():
            kp = (p + '.' + k) if p else k
            keypaths[kp]+=1
            if any(x in k.lower() for x in SUSPICIOUS) and not isinstance(v,(dict,list)):
                susp[kp][str(v)]+=1
            walk(v, kp)
    elif isinstance(o, list):
        for it in o: walk(it, p+'[]')
n=0
with open(DATA) as f:
    for line in f:
        line=line.strip()
        if line: walk(json.loads(line)); n+=1
print('scanned', n)
print('top-level key paths:', [k for k in sorted(keypaths) if '.' not in k and '[' not in k])
print('SUSPICIOUS (label-like) fields:', dict(susp) or 'NONE - no ground-truth label; we derive it.')

scanned 100000
top-level key paths: ['candidate_id', 'career_history', 'certifications', 'education', 'languages', 'profile', 'redrob_signals', 'skills']
SUSPICIOUS (label-like) fields: {'education[].tier': Counter({'tier_3': 53220, 'tier_4': 51885, 'tier_2': 27821, 'tier_1': 6852})}


## Step 1 — Stream the pool into an engineered feature table

One pass over the JSONL. Key features the JD cares about:
- `applied_ml_years` — tenure on jobs with an ML title **or** ML evidence in the description (broader than ML-title-at-product-only).
- `ir_assess` — best **verified** assessment score on a retrieval/IR/LLM skill (hard to fake).
- `retrieval_evidence`, `prod_evidence`, `eval_evidence` — prose signals.
- behavioral signals for the Phase-2 availability multiplier.

In [108]:
def count_hits(text, phrases): return sum(1 for ph in phrases if ph in text)

rows=[]
with open(DATA) as f:
    for line in f:
        line=line.strip()
        if not line: continue
        c=json.loads(line)
        p=c.get('profile',{}) or {}
        ch=c.get('career_history',[]) or []
        sig=c.get('redrob_signals',{}) or {}
        edu=c.get('education',[]) or []
        skills=c.get('skills',[]) or []
        cur_title=lc(p.get('current_title'))
        summary=lc(p.get('summary'))
        loc=lc(p.get('location')); ctry=lc(p.get('country'))
        all_desc=' '.join(lc(j.get('description')) for j in ch)
        all_titles=' '.join(lc(j.get('title')) for j in ch)+' '+cur_title
        company_text=' '.join(lc(j.get('company')) for j in ch)
        career_text=all_desc+' '+all_titles+' '+company_text
        text=summary+' '+all_desc+' '+all_titles
        sk_names=[lc(s.get('name')) for s in skills if isinstance(s,dict)]
        skill_text=' '.join(sk_names)
        durs=[j.get('duration_months') for j in ch if isinstance(j.get('duration_months'),(int,float))]
        avg_stint=(sum(durs)/len(durs)) if durs else None
        short_stints=sum(1 for d in durs if d and d<20)
        # Career mix by tenure: product-company operating context is a plus; service-only is a mild negative.
        applied_ml_months=0
        service_months=0
        product_months=0
        career_months=0
        for j in ch:
            t=lc(j.get('title')); jd=lc(j.get('description')); d=j.get('duration_months') or 0
            ind=lc(j.get('industry')); comp=lc(j.get('company'))
            career_months += d
            is_service = ind in SERVICE_INDS or any(cf in comp for cf in CONSULTING)
            is_product = ind in PRODUCT_INDS
            if is_service: service_months += d
            if is_product: product_months += d
            if any(m in t for m in ML_TITLE) or any(ph in jd for ph in ML_EVIDENCE): applied_ml_months+=d
        service_share = service_months / career_months if career_months else 0
        product_share = product_months / career_months if career_months else 0
        career_yoe = round(career_months/12, 2) if career_months else None
        profile_yoe = p.get('years_of_experience')
        starts=[parse_date(j.get('start_date')) for j in ch if parse_date(j.get('start_date'))]
        span_yoe = round((TODAY-min(starts)).days/365.25, 2) if starts else None
        # Ground YOE in dated career history and first-work span. A claimed YOE above the
        # first-job-start-to-today span is impossible, not just a weighting issue.
        yoe_gap_tenure = (profile_yoe - career_yoe) if (profile_yoe is not None and career_yoe is not None) else 0
        yoe_gap_span = (profile_yoe - span_yoe) if (profile_yoe is not None and span_yoe is not None) else 0
        profile_yoe_gt_span = bool(yoe_gap_span > 1.5)
        inflated_yoe = bool(yoe_gap_tenure > 1.5 or profile_yoe_gt_span)
        yoe_candidates=[v for v in [profile_yoe, career_yoe, span_yoe] if v is not None]
        yoe = round(min(yoe_candidates), 2) if inflated_yoe and yoe_candidates else profile_yoe
        service_only_career = bool(ch) and service_months == career_months and product_months == 0
        prior_india_work_signal = any(ci in career_text for ci in INDIA_CITIES) or 'india' in career_text or any(emp in company_text for emp in INDIA_EMPLOYER_HINTS)
        # verified retrieval/IR/LLM competency from assessment scores
        ass=sig.get('skill_assessment_scores',{}) or {}
        ass_values=list(ass.values())
        ir_scores=[v for k,v in ass.items() if any(tk in k.lower() for tk in IR_ASSESS_TOKENS)]
        ir_assess=max(ir_scores) if ir_scores else 0
        la=parse_date(sig.get('last_active_date'))
        days_inactive=(TODAY-la).days if la else None
        rows.append(dict(
            candidate_id=c['candidate_id'],
            yoe=yoe, profile_yoe=profile_yoe, career_yoe=career_yoe, span_yoe=span_yoe,
            yoe_gap_span=round(yoe_gap_span,2), profile_yoe_gt_span=profile_yoe_gt_span, inflated_yoe=inflated_yoe,
            current_title=cur_title,
            current_industry=lc(p.get('current_industry')),
            country=ctry, location=loc,
            india=(ctry=='india') or any(ci in loc for ci in INDIA_CITIES),
            prior_india_work_signal=prior_india_work_signal,
            jd_city=any(ci in loc for ci in JD_CITIES),
            n_jobs=len(ch), avg_stint_months=avg_stint, short_stints=short_stints,
            career_months=career_months, service_months=service_months, product_months=product_months,
            service_share=round(service_share,3), product_share=round(product_share,3),
            service_only_career=service_only_career,
            applied_ml_years=round(applied_ml_months/12,2),
            ir_assess=ir_assess,
            is_ml_title=any(m in cur_title for m in ML_TITLE),
            is_noneng_title=any(nt in cur_title for nt in NONENG_TITLES),
            consulting_only=bool(ch) and all(any(cf in lc(j.get('company')) for cf in CONSULTING) for j in ch),
            in_product_now=lc(p.get('current_industry')) in PRODUCT_INDS,
            ml_evidence=count_hits(text, ML_EVIDENCE),
            retrieval_evidence=count_hits(text, RETRIEVAL_EVIDENCE),
            vector_infra_evidence=count_hits(text, VECTOR_INFRA),
            prod_evidence=count_hits(text, PROD_EVIDENCE),
            eval_evidence=count_hits(text, EVAL_EVIDENCE),
            python_evidence=count_hits(text, PYTHON_EVIDENCE),
            nice_to_have_hits=count_hits(text, NICE_TO_HAVE),
            framework_demo_hits=count_hits(text, FRAMEWORK_DEMO),
            external_validation_hits=count_hits(text, EXTERNAL_VALIDATION),
            llm_recent_hits=count_hits(text, LLM_RECENT),
            research_hits=count_hits(text, RESEARCH_HINT),
            cv_speech_hits=count_hits(text, CV_SPEECH),
            nlp_present=('nlp' in text or 'retrieval' in text or 'language model' in text or 'information retrieval' in text),
            ai_skill_count=sum(1 for s in sk_names if any(k==s or k in s for k in AI_SKILL_KW)),
            n_skills=len(sk_names),
            edu_tier=(edu[0].get('tier') if edu else None),
            response_rate=sig.get('recruiter_response_rate'),
            days_inactive=days_inactive,
            open_to_work=sig.get('open_to_work_flag'),
            notice_days=sig.get('notice_period_days'),
            willing_relocate=sig.get('willing_to_relocate'),
            interview_completion=sig.get('interview_completion_rate'),
            offer_acceptance=sig.get('offer_acceptance_rate'),
            github=sig.get('github_activity_score'),
            completeness=sig.get('profile_completeness_score'),
            saved_30d=sig.get('saved_by_recruiters_30d'),
            assess_avg=(np.mean(ass_values) if ass_values else None),
            assess_max=(max(ass_values) if ass_values else None),
        ))
df=pd.DataFrame(rows)
print(df.shape)
df.head(3)

(100000, 58)


,candidate_id,yoe,profile_yoe,career_yoe,span_yoe,yoe_gap_span,profile_yoe_gt_span,inflated_yoe,current_title,current_industry,...,open_to_work,notice_days,willing_relocate,interview_completion,offer_acceptance,github,completeness,saved_30d,assess_avg,assess_max
0,CAND_0000001,6.9,6.9,6.83,6.99,-0.09,False,False,backend engineer,it services,...,True,60,False,0.71,0.58,9.2,86.9,4,49.725,64.8
1,CAND_0000002,12.5,12.5,12.42,12.43,0.07,False,False,operations manager,it services,...,True,60,False,0.62,-1.00,-1.0,78.7,10,NaN,NaN
2,CAND_0000003,1.1,1.1,1.08,1.16,-0.06,False,False,customer support,it services,...,False,150,False,0.86,-1.00,-1.0,31.9,4,NaN,NaN


## Step 2 — Distributions & sanity checks

In [109]:
num=['yoe','applied_ml_years','ir_assess','ml_evidence','retrieval_evidence','prod_evidence','eval_evidence','response_rate','days_inactive','notice_days','assess_avg']
display(df[num].describe().T)
print('India-based:', round(df.india.mean(),3), '| in JD cities:', round(df.jd_city.mean(),3), '| open_to_work:', round(df.open_to_work.mean(),3), '| willing_relocate:', round(df.willing_relocate.mean(),3))
print('have any IR assessment score:', int((df.ir_assess>0).sum()), '| ir_assess>=50:', int((df.ir_assess>=50).sum()))
print('Top industries:')
print(df.current_industry.value_counts().head(12))

,count,mean,std,min,25%,50%,75%,max
yoe,100000.0,7.163556,3.823853,0.67,3.90,6.80,9.90,15.00
applied_ml_years,100000.0,1.333190,2.013670,0.00,0.00,0.00,2.50,14.08
ir_assess,100000.0,2.256387,11.413023,0.00,0.00,0.00,0.00,97.30
ml_evidence,100000.0,0.578720,1.025443,0.00,0.00,0.00,1.00,12.00
retrieval_evidence,100000.0,0.010870,0.155281,0.00,0.00,0.00,0.00,6.00
prod_evidence,100000.0,0.603010,0.518914,0.00,0.00,1.00,1.00,8.00
eval_evidence,100000.0,0.019460,0.162547,0.00,0.00,0.00,0.00,7.00
response_rate,100000.0,0.436574,0.214122,0.02,0.25,0.44,0.62,0.95
days_inactive,100000.0,141.723890,66.588459,33.00,85.00,138.00,195.00,273.00
notice_days,100000.0,87.385800,36.589628,0.00,60.00,90.00,120.00,150.00


India-based: 0.751 | in JD cities: 0.292 | open_to_work: 0.353 | willing_relocate: 0.288
have any IR assessment score: 4062 | ir_assess>=50: 2514
Top industries:
current_industry
it services       29881
software          22417
manufacturing     22305
conglomerate       7571
paper products     7467
fintech            2808
food delivery      2514
e-commerce         1529
consulting         1274
edtech              610
saas                328
ai/ml               278
Name: count, dtype: int64


## Step 3 — The skills trap

The JD warns the right answer is *not* the most AI keywords. Verify skill *lists* are essentially **uniform random noise** (so only verified assessment scores carry signal).

In [110]:
skill_freq=collections.Counter()
with open(DATA) as f:
    for line in f:
        line=line.strip()
        if not line: continue
        for s in (json.loads(line).get('skills',[]) or []):
            if isinstance(s,dict): skill_freq[lc(s.get('name'))]+=1
print('Top-15 skill frequencies (note how flat ~ uniform):')
for k,v in skill_freq.most_common(15): print(f'  {k:<22} {v}')
vals=[v for _,v in skill_freq.most_common(60)]
print('Top-60 skills: min', min(vals), 'max', max(vals), 'spread', max(vals)-min(vals), '=> ~uniform sprinkle. Skill lists cannot rank.')

Top-15 skill frequencies (note how flat ~ uniform):
  html                   12246
  databricks             12244
  redux                  12222
  terraform              12187
  angular                12173
  figma                  12157
  salesforce crm         12157
  vue.js                 12142
  sales                  12138
  accounting             12136
  agile                  12135
  kafka                  12114
  excel                  12109
  bigquery               12108
  ci/cd                  12108
Top-60 skills: min 11841 max 12246 spread 405 => ~uniform sprinkle. Skill lists cannot rank.


## Step 4 — Trap families & honeypots

Quantify each deliberate trap family, then build a **precision-first** honeypot detector. The anachronism check now uses **word-boundary tokens** (fixes the `rag` ⊂ `storage` substring bug).

In [111]:
print('=== trap family counts ===')
print('keyword-stuffer (>=4 AI skills + non-eng title):', int(((df.ai_skill_count>=4)&(df.is_noneng_title)).sum()))
print('consulting-only career:', int(df.consulting_only.sum()))
print('service-only career:', int(df.service_only_career.sum()))
print('CV/speech without NLP:', int(((df.cv_speech_hits>=2)&(~df.nlp_present)).sum()))
print('research-heavy:', int((df.research_hits>=2).sum()))
print('recent-LLM-only (no ML background):', int(((df.llm_recent_hits>=1)&(df.applied_ml_years<1)&(df.ir_assess==0)&(df.ml_evidence<2)).sum()))

=== trap family counts ===
keyword-stuffer (>=4 AI skills + non-eng title): 4116
consulting-only career: 9745
service-only career: 9745
CV/speech without NLP: 0
research-heavy: 0
recent-LLM-only (no ML background): 33938


In [112]:
# Impossibility battery. All checks read COHERENT fields (dates/descriptions/summary/yoe);
# plus one verified rare skill-contradiction (expert proficiency with 0 months).
TOOL_BIRTH_EXT = {
    'sentence-transformers':2019,'sentence transformers':2019,'sbert':2019,'bge':2023,
    'e5':2022,'instructor embedding':2022,'openai embeddings':2022,'ada-002':2022,'text-embedding':2022,
    'pinecone':2021,'weaviate':2019,'qdrant':2021,'milvus':2019,'opensearch':2021,'faiss':2017,
    'colbert':2020,'splade':2021,'rag':2020,'lora':2021,'qlora':2023,'peft':2022,
    'langchain':2022,'llamaindex':2022,'llama index':2022,
    'chatgpt':2022,'gpt-4':2023,'gpt-3':2020,'gpt-3.5':2022,'llama':2023,'mistral':2023,'mixtral':2023,
    'vllm':2023,'stable diffusion':2022,'whisper':2022,'segment anything':2023,
}
YEARS_RE = re.compile(r'(\d+(?:\.\d+)?)\s*\+?\s*years?')

# Company-age honeypots: the spec explicitly mentions candidates with impossible tenure at
# companies founded later. Applied narrowly to high-fit JD-looking profiles to avoid sweeping up
# low-relevance synthetic background noise around startup names.
COMPANY_FOUNDED_AFTER = {
    'sarvam ai': dt.date(2023, 8, 1),
    'krutrim': dt.date(2023, 12, 1),
}
COMPANY_AGE_JD_TERMS = [
    'search','retrieval','ranking','recommendation','recsys','semantic search','vector search',
    'hybrid retrieval','embedding','candidate-jd','ndcg','mrr','offline/online','a/b testing',
    'production ml','learning-to-rank','nlp'
]
COMPANY_AGE_TITLES = ['senior','lead','staff','principal','search engineer','applied ml','nlp engineer']

def company_age_trap(c, job):
    co = lc(job.get('company'))
    founded = COMPANY_FOUNDED_AFTER.get(co)
    start = parse_date(job.get('start_date'))
    if not founded or not start or start >= founded:
        return None
    p = c.get('profile',{}) or {}
    title_text = lc(p.get('current_title')) + ' ' + lc(p.get('headline')) + ' ' + lc(job.get('title'))
    all_text = ' '.join([
        lc(p.get('headline')), lc(p.get('summary')), lc(job.get('title')), lc(job.get('description')),
        ' '.join(lc(j.get('description')) for j in (c.get('career_history',[]) or [])),
    ])
    jd_term_hits = count_hits(all_text, COMPANY_AGE_JD_TERMS)
    # Precision gate: only the convincing senior/search/applied/NLP profiles are honeypots.
    # Generic junior/data-science/CV-ish roles with impossible company dates are treated as background noise.
    senior_search_applied = any(t in title_text for t in ['senior','lead','staff','search engineer','applied ml','nlp engineer'])
    senior_ds = 'senior data scientist' in title_text
    if (senior_search_applied and jd_term_hits >= 7) or (senior_ds and jd_term_hits >= 6):
        return f'company_before_founding:{co}'
    return None

def exp_claim_anachronism(text):
    for m in YEARS_RE.finditer(text):
        n=float(m.group(1)); ctx=tokens(text[max(0,m.start()-60):m.end()+60])
        for tool,birth in TOOL_BIRTH_EXT.items():
            if tool_present(tool,ctx) and n > (2026-birth)+1: return 'exp_gt_tool_age:'+tool
    return None

def honeypot_reasons(c):
    r=[]
    p=c.get('profile',{}) or {}; ch=c.get('career_history',[]) or []
    y=p.get('years_of_experience') or 0
    if sum(1 for j in ch if j.get('is_current'))>1: r.append('two_current_jobs')
    for j in ch:
        s=parse_date(j.get('start_date')); e=parse_date(j.get('end_date')); dm=j.get('duration_months')
        if (s and s>TODAY) or (e and e>TODAY): r.append('future_date')
        if s and e and isinstance(dm,(int,float)):
            real=(e.year-s.year)*12+(e.month-s.month)
            if real<0 or abs(real-dm)>10: r.append('dur_mismatch')
        if y and isinstance(dm,(int,float)) and dm/12 > y+1.5: r.append('job_gt_yoe')
        d=lc(j.get('description')); toks=tokens(d)
        if e:
            for tool,birth in TOOL_BIRTH_EXT.items():
                if e.year<birth and tool_present(tool,toks): r.append('anachronism:'+tool); break
        cr = company_age_trap(c, j)
        if cr: r.append(cr)
        ec=exp_claim_anachronism(d)
        if ec: r.append(ec)
    ec=exp_claim_anachronism(lc(p.get('summary'))+' '+lc(p.get('headline')))
    if ec: r.append('summary_'+ec)
    dated=sorted([(parse_date(j.get('start_date')),parse_date(j.get('end_date'))) for j in ch
                  if parse_date(j.get('start_date')) and parse_date(j.get('end_date'))], key=lambda x:x[0])
    for i in range(len(dated)-1):
        s1,e1=dated[i]; s2,e2=dated[i+1]
        if s2<e1 and (min(e1,e2)-s2).days>365: r.append('job_overlap')
    durs=[j.get('duration_months') or 0 for j in ch]
    if y and sum(durs)/12 > y+5: r.append('tenure_gt_yoe')
    # expert proficiency claimed for a skill with 0 months of usage (rare, clean contradiction; 21 in pool)
    for sk in (c.get('skills') or []):
        if isinstance(sk,dict) and sk.get('proficiency')=='expert' and sk.get('duration_months')==0:
            r.append('expert_skill_0mo'); break
    return sorted(set(r))

hp_ids={}
with open(DATA) as f:
    for line in f:
        line=line.strip()
        if not line: continue
        c=json.loads(line); rs=honeypot_reasons(c)
        if rs: hp_ids[c['candidate_id']]=rs
print('after profile-level checks:', len(hp_ids))

# df-level impossibility: applied-ML tenure exceeds total stated experience
aml_imposs = df[(df.yoe.notna()) & (df.applied_ml_years > df.yoe.astype(float) + 1.5)]
for cid in aml_imposs.candidate_id:
    hp_ids.setdefault(cid, []).append('aml_gt_yoe')

print('high-confidence honeypot candidates flagged:', len(hp_ids))
rc=collections.Counter(r.split(':')[0] for rs in hp_ids.values() for r in rs)
print('reason breakdown:', dict(rc))
df['honeypot']=df.candidate_id.isin(hp_ids)


after profile-level checks: 76
high-confidence honeypot candidates flagged: 80
reason breakdown: {'anachronism': 32, 'expert_skill_0mo': 21, 'job_gt_yoe': 19, 'tenure_gt_yoe': 14, 'company_before_founding': 8, 'aml_gt_yoe': 6}


In [113]:
# ===== EXPORT every flagged honeypot with FULL data for manual inspection =====
hp_full=[]
with open(DATA) as f:
    for line in f:
        line=line.strip()
        if not line: continue
        c=json.loads(line)
        if c['candidate_id'] in hp_ids:
            c['_honeypot_reasons']=hp_ids[c['candidate_id']]
            hp_full.append(c)

# full JSON records (open this to see everything they had)
with open('honeypots_flagged.json','w') as fo:
    json.dump(hp_full, fo, indent=2)

# flat table + the triggering job description snippet
def trigger_snippet(c):
    for j in c.get('career_history',[]) or []:
        d=(j.get('description') or '')
        toks=tokens(d.lower())
        for tool,birth in TOOL_BIRTH_EXT.items():
            e=parse_date(j.get('end_date'))
            if e and e.year<birth and tool_present(tool,toks):
                return f"[{j.get('title')} @ {j.get('company')} ended {j.get('end_date')}] {d[:160]}"
    return ''

flat=[]
for c in hp_full:
    p=c.get('profile',{}) or {}
    flat.append(dict(candidate_id=c['candidate_id'], reasons='; '.join(c['_honeypot_reasons']),
        current_title=p.get('current_title'), industry=p.get('current_industry'),
        yoe=p.get('years_of_experience'), country=p.get('country'),
        evidence=trigger_snippet(c)))
hp_table=pd.DataFrame(flat).sort_values('candidate_id').reset_index(drop=True)
hp_table.to_csv('honeypots_flagged.csv', index=False)
print(f"wrote honeypots_flagged.json ({len(hp_full)} full records) + honeypots_flagged.csv")
pd.set_option('display.max_colwidth', 90)
display(hp_table[['candidate_id','reasons','current_title','yoe','evidence']])

wrote honeypots_flagged.json (80 full records) + honeypots_flagged.csv


,candidate_id,reasons,current_title,yoe,evidence
0,CAND_0001610,aml_gt_yoe,Machine Learning Engineer,3.0,
1,CAND_0002025,anachronism:qlora,Senior AI Engineer,5.9,[Lead AI Engineer @ Aganitha ended 2022-12-07] Fine-tuned LLaMA-2-7B and Mistral-7B va...
2,CAND_0003582,expert_skill_0mo,Mobile Developer,8.2,
3,CAND_0007353,job_gt_yoe; tenure_gt_yoe,Frontend Engineer,9.9,
4,CAND_0007412,anachronism:gpt-4; anachronism:openai embeddings,Applied ML Engineer,7.4,[Search Engineer @ Swiggy ended 2022-02-03] Implemented a RAG-based customer support c...
...,...,...,...,...,...
75,CAND_0095140,expert_skill_0mo,Backend Engineer,5.0,
76,CAND_0095317,expert_skill_0mo,HR Manager,7.0,
77,CAND_0095480,expert_skill_0mo,.NET Developer,2.3,
78,CAND_0096172,company_before_founding:krutrim,NLP Engineer,5.2,


In [114]:
# ===== VALIDATE the yoe-based checks: is years_of_experience consistent with career dates pool-wide? =====
# If mismatches are RARE -> the few are deliberate honeypots (exclude). If COMMON -> yoe is noise (don't flag).
res=[]
with open(DATA) as f:
    for line in f:
        line=line.strip()
        if not line: continue
        c=json.loads(line)
        y=c.get('profile',{}).get('years_of_experience')
        if y is None: continue
        ch=c.get('career_history') or []
        tenure=sum((j.get('duration_months') or 0) for j in ch)/12          # sum of job durations
        starts=[parse_date(j.get('start_date')) for j in ch if parse_date(j.get('start_date'))]
        span=((TODAY-min(starts)).days/365) if starts else 0                 # first job -> today
        res.append((y, tenure, span))
rr=pd.DataFrame(res, columns=['yoe','tenure','span'])
rr['tenure_minus_yoe']=rr.tenure-rr.yoe
rr['span_minus_yoe']=rr.span-rr.yoe
print('Across', len(rr), 'candidates:')
print('  median |tenure - yoe| :', round((rr.tenure-rr.yoe).abs().median(),2),'yrs')
print('  within +-1.5y (tenure vs yoe):', round((rr.tenure_minus_yoe.abs()<=1.5).mean()*100,1),'%')
print('  tenure exceeds yoe by >1.5y  :', round((rr.tenure_minus_yoe>1.5).mean()*100,2),'% of pool')
print('  median |span  - yoe| :', round((rr.span-rr.yoe).abs().median(),2),'yrs')
print('\n=> if "within +-1.5y" is high, yoe is normally CONSISTENT, so the few mismatches are deliberate.')

Across 100000 candidates:
  median |tenure - yoe| : 0.1 yrs
  within +-1.5y (tenure vs yoe): 100.0 %
  tenure exceeds yoe by >1.5y  : 0.02 % of pool
  median |span  - yoe| : 0.06 yrs

=> if "within +-1.5y" is high, yoe is normally CONSISTENT, so the few mismatches are deliberate.


## Step 5 — Derive the relevance-tier rubric (the labels)

Translate the JD fit-theory into a transparent **tier 0–5** function (our pseudo-label generator; calibrated in Phase 2). Now anchored on verified `ir_assess` + broader `applied_ml_years`.

| Tier | Meaning |
|---|---|
| **5** | Perfect: India/relocatable, 5–9 yrs, applied ML at product co, built ranking/search/recsys in production, eval experience, available |
| **4** | Strong, missing ~1 element |
| **3** | Relevant threshold: solid ML/data at product co with some retrieval/ranking exposure |
| **2** | Tangential |
| **1** | Weak / off-target |
| **0** | Irrelevant / disqualified (consulting-only, CV-only, research-only, non-eng title) |
| **-1** | Honeypot → excluded |

In [ ]:
# Shared scorer -> both the coarse tier (labels) and a continuous struct_score (for ranking)
def _fit(r):
    yoe=r['yoe'] or 0
    under_min_yoe = yoe < 4
    profile_yoe_gt_span = bool(r.get('profile_yoe_gt_span', False))
    consulting_only = bool(r['consulting_only'])
    cv_speech_without_nlp = bool(r['cv_speech_hits']>=2 and not r['nlp_present'])
    noneng_without_ml = bool(r['is_noneng_title'] and not r['is_ml_title'])
    research_without_prod = bool(r['research_hits']>=3 and r['prod_evidence']==0)

    disq = profile_yoe_gt_span or under_min_yoe or consulting_only or noneng_without_ml or cv_speech_without_nlp or research_without_prod

    s=0.0
    s += STRUCT_WEIGHTS['yoe_target'] if 5<=yoe<=9 else (STRUCT_WEIGHTS['yoe_greater_target'] if 9<yoe<=11 else (STRUCT_WEIGHTS['yoe_slightly_below_target'] if 4.4<=yoe<5 else (STRUCT_WEIGHTS['yoe_below_target'] if 4<=yoe<4.4 else STRUCT_WEIGHTS['yoe_far_below_target'])))
    s += min(r['applied_ml_years'],5)/5*STRUCT_WEIGHTS['applied_ml']

    # JD must-haves: production retrieval/search infra, strong Python, and ranking evaluation.
    s += STRUCT_WEIGHTS['python_strong'] if r['python_evidence']>=2 else (STRUCT_WEIGHTS['python_some'] if r['python_evidence']==1 else 0.0)
    s += STRUCT_WEIGHTS['ir_assess_strong'] if r['ir_assess']>=50 else (STRUCT_WEIGHTS['ir_assess_some'] if r['ir_assess']>0 else 0.0)
    retrieval_prod_hits = r['retrieval_evidence'] + r['prod_evidence']
    s += STRUCT_WEIGHTS['retrieval_prod_strong'] if retrieval_prod_hits>=3 else (STRUCT_WEIGHTS['retrieval_prod_some'] if retrieval_prod_hits>=1 else 0.0)
    s += STRUCT_WEIGHTS['vector_infra_strong'] if r['vector_infra_evidence']>=2 else (STRUCT_WEIGHTS['vector_infra_some'] if r['vector_infra_evidence']==1 else 0.0)
    s += STRUCT_WEIGHTS['ranking_eval'] if r['eval_evidence']>=1 else 0.0

    # Useful, but not decisive by themselves.
    s += STRUCT_WEIGHTS['product_context'] if r['in_product_now'] else 0.0
    if r['product_months'] > r['service_months'] and r['product_months'] > 0:
        s += STRUCT_WEIGHTS['product_heavy_career']
    elif r['service_months'] > r['product_months'] and r['product_months'] > 0:
        s += STRUCT_WEIGHTS['service_heavy_with_product']
    completeness = r['completeness'] if pd.notna(r['completeness']) else None
    if completeness is None:
        pass
    elif completeness > 70:
        s += STRUCT_WEIGHTS['profile_complete_high']
    elif completeness >= 50:
        s += STRUCT_WEIGHTS['profile_complete_mid']
    elif completeness >= 30:
        s += STRUCT_WEIGHTS['profile_complete_low']
    else:
        s += STRUCT_WEIGHTS['profile_complete_floor']
    assess_quality = r['assess_avg'] if pd.notna(r['assess_avg']) else 0
    if assess_quality >= 80:
        s += STRUCT_WEIGHTS['assessment_quality_high']
    elif assess_quality >= 60:
        s += STRUCT_WEIGHTS['assessment_quality_mid']
    elif assess_quality >= 40:
        s += STRUCT_WEIGHTS['assessment_quality_low']
    s += STRUCT_WEIGHTS['ml_title'] if r['is_ml_title'] else 0.0
    s += min(r['nice_to_have_hits'], int(round(STRUCT_WEIGHTS['nice_cap']/STRUCT_WEIGHTS['nice_each']))) * STRUCT_WEIGHTS['nice_each']

    # Mild location signal here; hard logistics feasibility is handled in ranker.ipynb.
    if r['jd_city']: s += LOCATION_STRUCT_WEIGHTS['jd_city']
    elif r['india'] and r['willing_relocate']: s += LOCATION_STRUCT_WEIGHTS['india_relocate']
    elif r['india']: s += LOCATION_STRUCT_WEIGHTS['india']
    elif r['willing_relocate']: s += LOCATION_STRUCT_WEIGHTS['outside_relocate']
    else: s += LOCATION_STRUCT_WEIGHTS['outside_no_relocate']

    # Explicit JD anti-fit patterns. Consulting-only/CV-only/research-only are also hard tier-0 gates.
    title_chaser = bool(r['short_stints']>=3 and (yoe>=5 or r['avg_stint_months'] and r['avg_stint_months']<24))
    framework_demo_only = bool(r['framework_demo_hits']>=2 and r['retrieval_evidence']==0 and r['vector_infra_evidence']==0 and r['prod_evidence']==0 and r['eval_evidence']==0)
    closed_proprietary_no_validation = bool((r['applied_ml_years']>=5 or yoe>=7) and r['external_validation_hits']==0 and (r['github'] or 0)<30 and r['eval_evidence']==0)
    recent_llm_only = bool(r['llm_recent_hits']>=1 and r['applied_ml_years']<1 and r['ir_assess']==0 and r['ml_evidence']<2)

    if title_chaser: s += STRUCT_PENALTIES['title_chaser']
    if framework_demo_only: s += STRUCT_PENALTIES['framework_demo_only']
    if r['service_only_career']: s += STRUCT_PENALTIES['service_only_career']
    if consulting_only: s += STRUCT_PENALTIES['consulting_only']
    if cv_speech_without_nlp: s += STRUCT_PENALTIES['cv_speech_without_nlp']
    if closed_proprietary_no_validation: s += STRUCT_PENALTIES['closed_proprietary_no_validation']
    if research_without_prod: s += STRUCT_PENALTIES['research_without_prod']
    if recent_llm_only: s += STRUCT_PENALTIES['recent_llm_only']
    return s, disq

def tier(r):
    if r['honeypot']: return -1
    s,disq=_fit(r)
    if disq: return 0
    if s>=6.2: return 5
    if s>=4.8: return 4
    if s>=3.4: return 3
    if s>=2.2: return 2
    return 1

def struct_score(r):
    if r['honeypot']: return 0.0
    s,disq=_fit(r)
    return 0.0 if disq else max(s,0.0)

df['tier']=df.apply(tier,axis=1)
df['struct_score']=df.apply(struct_score,axis=1)
print('Tier distribution:')
print(df.tier.value_counts().sort_index())
print('relevant (tier>=3):', int((df.tier>=3).sum()), '| honeypots excluded:', int((df.tier==-1).sum()))
print('struct_score range:', round(df.struct_score.min(),2), '..', round(df.struct_score.max(),2))
# location-bucket sanity
print('\nlocation buckets:')
print('  in JD/Tier-1 city            :', int(df.jd_city.sum()))
print('  India non-preferred + relocate:', int((df.india & ~df.jd_city & df.willing_relocate).sum()))
print('  India non-preferred + NO move :', int((df.india & ~df.jd_city & ~df.willing_relocate).sum()))


Tier distribution:
tier
-1       80
 0    67911
 1     7450
 2    10599
 3     9413
 4     2128
 5     2419
Name: count, dtype: int64
relevant (tier>=3): 13960 | honeypots excluded: 80
struct_score range: 0.0 .. 12.2

location buckets:
  in JD/Tier-1 city            : 29231
  India non-preferred + relocate: 13140
  India non-preferred + NO move : 32742


In [116]:
for t in [5,4,3,2,1,0]:
    sub=df[df.tier==t].head(2)
    print('===== TIER', t, '=====')
    for _,r in sub.iterrows():
        print(f'  {r.candidate_id} | {r.current_title} | {r.current_industry} | yoe={r.yoe} aml_yrs={r.applied_ml_years} ir={r.ir_assess} retr={r.retrieval_evidence} india={r.india} resp={r.response_rate}')

===== TIER 5 =====
  CAND_0000010 | data engineer | transportation | yoe=4.6 aml_yrs=4.58 ir=0.0 retr=0 india=False resp=0.4
  CAND_0000014 | frontend engineer | food delivery | yoe=8.4 aml_yrs=0.0 ir=77.6 retr=0 india=True resp=0.8
===== TIER 4 =====
  CAND_0000088 | java developer | it services | yoe=7.6 aml_yrs=0.0 ir=73.1 retr=0 india=True resp=0.53
  CAND_0000110 | software engineer | food delivery | yoe=9.8 aml_yrs=0.0 ir=65.2 retr=0 india=True resp=0.67
===== TIER 3 =====
  CAND_0000001 | backend engineer | it services | yoe=6.9 aml_yrs=4.58 ir=41.6 retr=0 india=False resp=0.34
  CAND_0000015 | software engineer | fintech | yoe=5.4 aml_yrs=0.0 ir=0.0 retr=0 india=True resp=0.32
===== TIER 2 =====
  CAND_0000007 | civil engineer | it services | yoe=5.5 aml_yrs=0.0 ir=0.0 retr=0 india=True resp=0.62
  CAND_0000020 | mechanical engineer | it services | yoe=6.3 aml_yrs=0.5 ir=0.0 retr=0 india=True resp=0.55
===== TIER 1 =====
  CAND_0000009 | mechanical engineer | paper products | y

In [117]:
# ====== ADVERSARIAL SELF-AUDIT: did our POSITIVE scorer promote impossible profiles? ======
# Score every honeypot AS IF it were genuine (ignore the exclusion) and see what tier
# our fit-logic would have given it. Tier>=4 means our system called a fabricated
# profile a 'great candidate' -- exactly the failure we must catch.
def tier_ignore_hp(row):
    d=dict(row); d['honeypot']=False
    return tier(d)

hp=df[df.honeypot].copy()
hp['would_be_tier']=hp.apply(tier_ignore_hp, axis=1)
print('What tier our FIT logic would have given the', len(hp), 'honeypots (if not excluded):')
print(hp['would_be_tier'].value_counts().sort_index())
promoted=hp[hp.would_be_tier>=4]
print(f'\n>>> {len(promoted)} honeypots our scorer rates Tier 4-5 (would have landed in our shortlist):')
for _,r in promoted.sort_values('would_be_tier',ascending=False).head(12).iterrows():
    rs=hp_ids[r.candidate_id]
    print(f'  t{r.would_be_tier} {r.candidate_id} | {r.current_title} | {r.current_industry} | yoe={r.yoe} aml={r.applied_ml_years} ir={r.ir_assess} | IMPOSSIBLE: {rs}')

# also: of our genuine Tier-5, how many lean on a SINGLE feature? (fragile positives)
print('\n--- fragility of genuine Tier-5 (non-honeypot) ---')
t5=df[df.tier==5]
single_ir = int(((t5.ir_assess>=50)&(t5.applied_ml_years<1)&(t5.retrieval_evidence==0)).sum())
print(f'Tier-5 total: {len(t5)} | rely on ir_assess alone (no ML career, no retrieval prose): {single_ir}')

What tier our FIT logic would have given the 80 honeypots (if not excluded):
would_be_tier
0    38
1     2
2     3
3     4
5    33
Name: count, dtype: int64

>>> 33 honeypots our scorer rates Tier 4-5 (would have landed in our shortlist):
  t5 CAND_0002025 | senior ai engineer | consumer electronics | yoe=5.9 aml=5.83 ir=0.0 | IMPOSSIBLE: ['anachronism:qlora']
  t5 CAND_0046064 | senior nlp engineer | software | yoe=8.9 aml=8.83 ir=0.0 | IMPOSSIBLE: ['anachronism:pinecone']
  t5 CAND_0096172 | nlp engineer | ai/ml | yoe=5.2 aml=5.08 ir=0.0 | IMPOSSIBLE: ['company_before_founding:krutrim']
  t5 CAND_0092278 | senior nlp engineer | software | yoe=6.8 aml=6.67 ir=0.0 | IMPOSSIBLE: ['anachronism:bge']
  t5 CAND_0080766 | staff machine learning engineer | software | yoe=8.8 aml=8.75 ir=0.0 | IMPOSSIBLE: ['anachronism:sentence-transformers']
  t5 CAND_0079064 | senior data scientist | healthtech ai | yoe=5.2 aml=5.17 ir=0.0 | IMPOSSIBLE: ['anachronism:gpt-4']
  t5 CAND_0077285 | recommendati

In [118]:
# ===== LOCATION vs COUNTRY consistency: can't be in an Indian city AND a foreign country =====
loc = df.location.fillna('').str.lower()
# quick peek at how location strings look per country
print('sample locations by country:')
for ctry in ['india','usa','canada','uk','germany','singapore','uae','australia']:
    s=df[df.country==ctry].location.dropna().unique()[:4]
    print(f'  {ctry:10}: {list(s)}')

FOREIGN_CITIES={'toronto','austin','london','berlin','singapore','dubai','sydney','munich','manchester',
                'vancouver','melbourne','san francisco','new york','seattle','boston','chicago','dublin',
                'amsterdam','abu dhabi','frankfurt','perth','brisbane','glasgow','birmingham'}
df['loc_has_india_city']  = loc.apply(lambda L: any(ci in L for ci in INDIA_CITIES))
df['loc_has_foreign_city']= loc.apply(lambda L: any(ci in L for ci in FOREIGN_CITIES))

a = df[df.loc_has_india_city & (df.country!='india')]
b = df[df.loc_has_foreign_city & (df.country=='india')]
print('\n>>> Indian city in location but country != India:', len(a), '| of those tier>=3:', int((a.tier>=3).sum()))
print(a.country.value_counts().to_dict())
print(a[['candidate_id','location','country','current_title','tier']].head(15).to_string(index=False))
print('\n>>> Foreign city in location but country == India:', len(b), '| of those tier>=3:', int((b.tier>=3).sum()))
print(b[['candidate_id','location','country','current_title','tier']].head(15).to_string(index=False))

sample locations by country:
  india     : ['chennai, tamil nadu', 'gurgaon, haryana', 'noida, uttar pradesh', 'hyderabad, telangana']
  usa       : ['austin', 'new york', 'san francisco', 'seattle']
  canada    : ['toronto']
  uk        : ['london']
  germany   : ['berlin']
  singapore : ['singapore']
  uae       : ['dubai']
  australia : ['sydney']

>>> Indian city in location but country != India: 0 | of those tier>=3: 0
{}
Empty DataFrame
Columns: [candidate_id, location, country, current_title, tier]
Index: []

>>> Foreign city in location but country == India: 0 | of those tier>=3: 0
Empty DataFrame
Columns: [candidate_id, location, country, current_title, tier]
Index: []


In [119]:
# ===== JD's keyword-stuffer trap: 'all AI keywords as skills but title = Marketing Manager' =====
# This is a FIT problem (a real but unsuitable person), NOT an impossibility ->
# it must be handled by the tier/ranking, not honeypot exclusion. Verify our logic sinks them.
stuffers = df[df.is_noneng_title & (df.ai_skill_count>=5)]
print('non-eng-title candidates with >=5 AI skills in their (noise) skill list:', len(stuffers))
print('their TIER distribution (expect all 0 = correctly NOT a fit):')
print(stuffers.tier.value_counts().sort_index())
print('\nexamples (note the high AI-skill counts, yet correctly Tier 0):')
print(stuffers[['candidate_id','current_title','current_industry','ai_skill_count','tier']].head(10).to_string(index=False))

# the genuinely-suspicious version your instinct points at: title says non-eng, but the
# CAREER DESCRIPTIONS (not the noise skills) claim heavy ML/retrieval work = title-vs-work incoherence
incoherent = df[df.is_noneng_title & (~df.is_ml_title) & (df.retrieval_evidence>=1)]
print('\nnon-eng title but career PROSE claims retrieval work (semantic-incoherence candidates):', len(incoherent))
print(incoherent[['candidate_id','current_title','retrieval_evidence','applied_ml_years','tier']].head(10).to_string(index=False))

non-eng-title candidates with >=5 AI skills in their (noise) skill list: 4116
their TIER distribution (expect all 0 = correctly NOT a fit):
tier
0    4116
Name: count, dtype: int64

examples (note the high AI-skill counts, yet correctly Tier 0):
candidate_id      current_title current_industry  ai_skill_count  tier
CAND_0000021    project manager      it services               6     0
CAND_0000074 operations manager   paper products              10     0
CAND_0000083   graphic designer      it services              10     0
CAND_0000120   graphic designer     conglomerate              10     0
CAND_0000133   graphic designer    manufacturing               7     0
CAND_0000201  marketing manager    manufacturing               6     0
CAND_0000203 operations manager     conglomerate               6     0
CAND_0000211 operations manager      it services               9     0
CAND_0000220  marketing manager   paper products               9     0
CAND_0000312     content writer      it serv

In [120]:
# ===== BEHAVIORAL AVAILABILITY MULTIPLIER (JD directive #3) =====
# Fit answers 'are they good?'. This answers 'can we actually talk to them?'.
# Location, relocation, and notice period are handled separately by the ranker logistics multiplier.
# Grounded in pool distributions: response_rate median 0.44; days_inactive 33..273 (median 138);
# the JD calls out '6 months inactive (~180d) + 5% response' as effectively unavailable.
def behavioral_multiplier(r):
    resp = r['response_rate'] if pd.notna(r['response_rate']) else 0.0
    if resp > 0.60:
        response_factor = 0.70
    elif resp >= 0.40:
        response_factor = 0.40
    else:
        response_factor = 0.10
    di   = r['days_inactive'] if pd.notna(r['days_inactive']) else 220
    recency = max(0.0, min(1.0, (270 - di)/(270-40)))   # ~1 if active recently, ->0 by ~9 months
    otw  = 1.0 if r['open_to_work'] else 0.0
    return round(response_factor + 0.20*recency + 0.10*otw, 3)   # range ~0.10 .. 1.0

df['behavioral'] = df.apply(behavioral_multiplier, axis=1)
print('behavioral multiplier distribution:')
print(df['behavioral'].describe()[['min','25%','50%','75%','max']].round(3).to_string())

jd_case = behavioral_multiplier({'response_rate':0.05,'days_inactive':190,'open_to_work':False})
ideal   = behavioral_multiplier({'response_rate':0.9,'days_inactive':45,'open_to_work':True})
print(f"\nJD's 'unavailable' candidate (5% resp, ~6mo idle, not looking) -> multiplier {jd_case}")
print(f"Reachable candidate (90% resp, active, open-to-work)       -> multiplier {ideal}")

# effect within our Tier-5: who is actually reachable?
t5=df[df.tier==5].sort_values('behavioral',ascending=False)
cols=['candidate_id','current_title','response_rate','days_inactive','open_to_work','notice_days','behavioral']
print('\nMost reachable Tier-5:'); print(t5[cols].head(3).to_string(index=False))
print('Least reachable Tier-5 (down-weighted despite perfect fit):'); print(t5[cols].tail(3).to_string(index=False))


behavioral multiplier distribution:
min    0.100
25%    0.259
50%    0.477
75%    0.737
max    1.000

JD's 'unavailable' candidate (5% resp, ~6mo idle, not looking) -> multiplier 0.17
Reachable candidate (90% resp, active, open-to-work)       -> multiplier 0.996

Most reachable Tier-5:
candidate_id current_title  response_rate  days_inactive  open_to_work  notice_days  behavioral
CAND_0068932   ml engineer           0.82             34          True           30         1.0
CAND_0028527 ai specialist           0.90             35          True           90         1.0
CAND_0012065  data analyst           0.72             38          True           90         1.0
Least reachable Tier-5 (down-weighted despite perfect fit):
candidate_id                    current_title  response_rate  days_inactive  open_to_work  notice_days  behavioral
CAND_0084602             full stack developer           0.36            193         False          150       0.167
CAND_0041611  staff machine learning en

In [121]:
# ===== Company-name analysis: are fictional companies (Stark Industries, Hooli...) a fake signal? =====
# (runs after tier is computed)
comp=collections.Counter()
FICTIONAL={'dunder mifflin','pied piper','hooli','aviato','raviga','stark industries','wayne enterprises',
 'lexcorp','initech','oscorp','globex','umbrella corporation','cyberdyne systems','cyberdyne','wonka industries',
 'acme corporation','acme','massive dynamic','gringotts','ingen','weyland-yutani','ocp','planet express','momcorp',
 'los pollos hermanos','madrigal','e corp','evil corp','allsafe','lumon','soylent','tyrell corporation','tyrell',
 'nakatomi','prestige worldwide','vandelay industries','vandelay','sterling cooper','wernham hogg','wonka',
 'gekko','duff','bluth','genco'}
rows_fic=[]
with open(DATA) as f:
    for line in f:
        line=line.strip()
        if not line: continue
        c=json.loads(line)
        comps=[(j.get('company') or '') for j in (c.get('career_history') or [])]
        comps.append((c.get('profile',{}) or {}).get('current_company') or '')
        comps=[x for x in comps if x]
        for x in comps: comp[x]+=1
        rows_fic.append((c['candidate_id'], any(x.lower() in FICTIONAL for x in comps)))

print('TOP 40 company names in the pool:')
for k,v in comp.most_common(40): print(f'  {v:6}  {k}')

fic=pd.DataFrame(rows_fic, columns=['candidate_id','has_fictional']).set_index('candidate_id')
d2=df.set_index('candidate_id').join(fic)
print('\ncandidates with >=1 known-fictional company:', int(d2.has_fictional.sum()),
      f'({d2.has_fictional.mean()*100:.1f}% of pool)')
print('fictional-company RATE by tier (flat across tiers => dataset flavor, NOT a signal):')
print(d2.groupby('tier').has_fictional.mean().round(3).to_string())

TOP 40 company names in the pool:
   31312  Infosys
   31248  Wipro
   31127  Wayne Enterprises
   31118  Initech
   31114  Pied Piper
   31036  Acme Corp
   30963  Globex Inc
   30934  TCS
   30887  Hooli
   30883  Dunder Mifflin
   30847  Stark Industries
    4307  Swiggy
    4172  Razorpay
    4165  CRED
    4160  Capgemini
    4145  Accenture
    4144  HCL
    4109  Zomato
    4104  Mindtree
    4076  Cognizant
    4053  Flipkart
    4005  Tech Mahindra
    3975  Mphasis
     570  Meesho
     550  Nykaa
     548  InMobi
     517  Zoho
     514  Ola
     513  Vedantu
     512  BYJU'S
     510  PolicyBazaar
     504  Paytm
     501  Freshworks
     493  upGrad
     486  PharmEasy
     480  PhonePe
     478  Dream11
     476  Unacademy
     123  Genpact AI
     112  Glance

candidates with >=1 known-fictional company: 72705 (72.7% of pool)
fictional-company RATE by tier (flat across tiers => dataset flavor, NOT a signal):
tier
-1    0.375
 0    0.729
 1    0.837
 2    0.759
 3    0.74

In [122]:
# ===== EXPORT feature table for the ranker (ranker.ipynb loads this) =====
import os
os.makedirs('artifacts', exist_ok=True)
keep=['candidate_id','tier','honeypot','struct_score','behavioral','yoe','profile_yoe','career_yoe','span_yoe','yoe_gap_span','profile_yoe_gt_span','inflated_yoe','current_title',
      'current_industry','country','location','india','prior_india_work_signal','jd_city','willing_relocate','applied_ml_years','ir_assess',
      'career_months','service_months','product_months','service_share','product_share','service_only_career',
      'retrieval_evidence','vector_infra_evidence','prod_evidence','eval_evidence','python_evidence',
      'nice_to_have_hits','framework_demo_hits','external_validation_hits','is_ml_title','in_product_now',
      'completeness','assess_avg','assess_max',
      'response_rate','days_inactive','open_to_work','notice_days']
df[keep].to_parquet('artifacts/features.parquet', index=False)
print('saved artifacts/features.parquet', df[keep].shape)

saved artifacts/features.parquet (100000, 45)


## Findings → next steps

- **No ground-truth label in the data** → we generate pseudo-tiers from the JD fit-theory (Step 5).
- **Skills lists are uniform noise** → ranker ignores them for positive fit; **verified `skill_assessment_scores` (IR/LLM)** are the gold positive signal.
- **Real signal = career-history prose** (retrieval/ranking/production evidence), so phrase counters and embeddings both read work history rather than keyword lists.
- **Honeypots** flagged precision-first with word-boundary anachronism detection.
- **Behavioral signals** become an availability multiplier (Phase 2), not part of the tier itself.

**Phase 2:** add the semantic (embedding) relevance score, blend with this structured tier, apply the behavioral multiplier, and calibrate weights against an LLM-labeled sample.